In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter
import imageio.v2 as imageio
from numba import jit
from io import BytesIO

In [ ]:
@jit(nopython=True)
def get_neighbor_sum(spins, L, x, y):
  return (spins[(x + 1) % L, y] +
          spins[(x - 1) % L, y] +
          spins[x, (y + 1) % L] +
          spins[x, (y - 1) % L])

@jit(nopython = True)
def metropolis_step(spins, L, J, beta):
  x, y = np.random.randint(0, L, 2)

  delta = 2 * J * get_neighbor_sum(spins, L, x, y) * spins[x, y]
    
  if delta <= 0 or np.random.random() < np.exp(-beta * delta):
    spins[x, y] = -spins[x, y]


@jit(nopython = True)
def kawabata_step(spins, L, J, beta):
  x, y = np.random.randint(0, L, 2)

  d = np.random.randint(0, 4)
  if d == 0:   nx, ny = (x + 1) % L, y
  elif d == 1: nx, ny = (x - 1) % L, y
  elif d == 2: nx, ny = x, (y + 1) % L
  else:        nx, ny = x, (y - 1) % L

  S_A = spins[x, y]
  S_B = spins[nx, ny]

  if S_A == S_B:
    return

  delta = 2 * J * (S_A * (get_neighbor_sum(spins, L, x, y) - S_B) +
                   S_B * (get_neighbor_sum(spins, L, nx, ny) - S_A))
    
  if delta <= 0 or np.random.random() < np.exp(-beta * delta):
    spins[x, y] = S_B
    spins[nx, ny] = S_A


@jit(nopython=True)
def run_sweep(spins, L, J, beta, mode=0):
  # mode=0 -> Metropolis, mode=1 -> Kawabata
  for _ in range(L**2):
    if mode == 0:
      metropolis_step(spins, L, J, beta)
    else:
      kawabata_step(spins, L, J, beta)

In [ ]:
def spatial_correlation(spins, sigma):
  field = gaussian_filter(spins.astype(float), sigma=sigma, mode='wrap')
  
  grad_x = np.gradient(field, axis=1)
  grad_y = np.gradient(field, axis=0)
  
  gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
  
  return gradient_magnitude


def correlation(spins, shift):
  shifted_x = np.roll(spins, shift, axis=1) + np.roll(spins, -shift, axis=1)
  shifted_y = np.roll(spins, shift, axis=0) + np.roll(spins, -shift, axis=0)

  return np.mean(spins * (shifted_x + shifted_y) / 4)


def correlation_function(spins, L):
  correlations = []
  for r in range(L // 2):
    corr_x = np.mean(spins * np.roll(spins, -r, axis=0))
    corr_y = np.mean(spins * np.roll(spins, -r, axis=1))
    chi_r = (corr_x + corr_y) / 2
    correlations.append(chi_r)
    
  return np.array(correlations)


def model(t, a, b):
  return a * (t ** b)


def calculate_domain_size(chi_values):
  cutoff = 0.3
  valid_indices = np.where(chi_values > cutoff)[0]    
  r_max = valid_indices[-1]
  
  chi_subset = chi_values[:r_max + 1]
  log_sum = np.sum(np.log(np.abs(chi_subset)))
  
  R = (r_max * (r_max + 1)) / (-2 * log_sum)
  return R


def run_experiment(L, J, beta, steps_list, algorithm_name="metropolis"):
  spins = np.random.choice([-1, 1], (L, L))
  mode = 1 if algorithm_name.lower() == "kawabata" else 0
  
  measured_R = []
  current_time = 0
  
  for target_time in steps_list:
    steps_to_do = target_time - current_time
    
    for _ in range(steps_to_do):
      run_sweep(spins, L, J, beta, mode)    
    current_time = target_time
    
    chi = correlation_function(spins, L)
    R = calculate_domain_size(chi)
    measured_R.append(R)

  return np.array(steps_list), np.array(measured_R)


CONFIG = {
  'J': 1,
  'beta': 0.5,
  'L_metropolis': 500,
  'L_kawabata': 200,
  'steps_metropolis': [1, 10, 100, 500, 1000, 2000, 3300, 5000],
  'steps_kawabata': [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000],
}

In [ ]:
def draw_spins(ax, spins, L):
  ax.imshow(spins, cmap='gray')

def draw_borders(ax, spins, L):
  ax.imshow(spatial_correlation(spins, 2), cmap='gray_r')

def draw_correlation(ax, spins, L):
  ax.set_aspect('auto')
  correl = np.zeros(L // 2)
  for shift in range(L // 2):
    correl[shift] = np.abs(correlation(spins, shift))
  
  ax.plot(range(1, L // 2), correl[1:], 'o-', label=r'$R_0$')
  ax.set_title("Correlation")
  ax.set_xlabel(r'$r$')
  ax.set_ylabel("Correlation")
  ax.set_ylim(0, 1)
  ax.legend()

def run_animation(filename, draw_func, mode_name, steps_to_save, L, J, beta, duration=0.4):
  print(f"Generowanie: {filename}")
  spins = np.random.choice([-1, 1], (L, L))
  mode = 1 if mode_name.lower() == "kawabata" else 0
  
  fig, ax = plt.subplots(figsize=(6, 6))
  ax.set_aspect('equal')
  ax.axis("off")
  
  with imageio.get_writer(filename, mode='I', duration=duration) as writer:
    for i in range(max(steps_to_save) + 1):
      run_sweep(spins, L, J, beta, mode)
      
      if i in steps_to_save:
        ax.clear()
        draw_func(ax, spins, L)
        
        buf = BytesIO()
        fig.savefig(buf, format='png')
        buf.seek(0)
        writer.append_data(imageio.imread(buf))
              
  plt.close(fig)

anim_tasks = [
  ('../results/gifs/08_metropolis.gif',             draw_spins,       'metropolis', CONFIG['steps_metropolis'], CONFIG['L_metropolis']),
  ('../results/gifs/08_metropolis_borders.gif',     draw_borders,     'metropolis', CONFIG['steps_metropolis'], CONFIG['L_metropolis']),
  ('../results/gifs/08_metropolis_correlation.gif', draw_correlation, 'metropolis', CONFIG['steps_metropolis'], CONFIG['L_metropolis']),
  ('../results/gifs/08_kawabata.gif',               draw_spins,       'kawabata',   CONFIG['steps_kawabata'],   CONFIG['L_kawabata']),
  ('../results/gifs/08_kawabata_borders.gif',       draw_borders,     'kawabata',   CONFIG['steps_kawabata'],   CONFIG['L_kawabata']),
  ('../results/gifs/08_kawabata_correlation.gif',   draw_correlation, 'kawabata',   CONFIG['steps_kawabata'],   CONFIG['L_kawabata']),
]

for filename, func, mode, steps, length in anim_tasks:
  run_animation(filename, func, mode, steps, length, CONFIG['J'], CONFIG['beta'])

In [ ]:
t_met, r_met = run_experiment(CONFIG['L_metropolis'], CONFIG['J'], CONFIG['beta'], CONFIG['steps_metropolis'], "metropolis")
t_kaw, r_kaw = run_experiment(CONFIG['L_kawabata'],   CONFIG['J'], CONFIG['beta'], CONFIG['steps_kawabata'],   "kawabata")

plt.figure(figsize=(10, 7))

def plot_kinetics(t, r, color, label_prefix, theoretical_exp):
  popt, _ = curve_fit(model, t, r)
  fit_label = f'{label_prefix} Fit: $t^{{{popt[1]:.3f}}}$'
  plt.loglog(t, model(t, *popt), f'{color}--', label=fit_label)

  plt.loglog(t, r, f'{color}o' if label_prefix=='Metropolis' else f'{color}s', label=f'{label_prefix} Data')
  
  last_t, last_r = t[-1], r[-1]
  plt.loglog(t, t**theoretical_exp * (last_r/last_t**theoretical_exp), f'{color}:', alpha=0.5)

plot_kinetics(t_met, r_met, 'b', 'Metropolis', 0.5)
plot_kinetics(t_kaw, r_kaw, 'r', 'Kawabata', 1/3)

plt.xlabel('Time (MCS)')
plt.ylabel('Typical Domain Size $R$')
plt.title('Domain Growth Kinetics in 2D Ising Model: Metropolis vs Kawabata')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.3)
plt.show()